In [2]:
import os

# Define the path to the documents
documents_path = "/Users/jayeshbhadane/Desktop/vettura-genai/Codes/Week_3/Day_1/Documents_For_Hybrid_Search_Assignment"

# Load all documents into a list
documents = []
for filename in os.listdir(documents_path):
    if filename.endswith(".txt"):  # Assuming the documents are text files
        with open(os.path.join(documents_path, filename), "r", encoding="utf-8") as file:
            content = file.read()
            documents.append({"filename": filename, "content": content})

print(f"Loaded {len(documents)} documents.")
# print(documents)

Loaded 5 documents.


In [3]:
documents[2]

{'filename': 'ISL28177.txt',
 'content': 'Description:\nThe ISL28177 is an OP07 replacement featuring low input offset voltage, low input bias current, and competitive noise and AC performance. The ESD ratings are best among competitive parts at 5kV HBM, 300V MM, and 2. 2kV CDM. The amplifier operates over the 6V (±3V) to 40V (±20V) range. Applications include precision active filters, medical and analytical instrumentation, precision power supply controls, and industrial sensors. The ISL28177 is available in the SOT23-5 and SOIC-8 packages and operates over the extended temperature range to -40°C to +125°C.\n\nFeatures:\nWide Supply Range: 6V (±3V) to 40V (±20V)\nLow Input Offset Voltage: 150µV, Max\nInput Bias Current: 1nA, Max\nLow Noise: 9.5nV/√Hz @ 1kHz\nGain Bandwidth: 600kHz\nExceptional ESD Performance: 5kV HBM, 300V MM, 2.2kV CDM\nOperating Temperature Range: -40°C to +125°C\nPackages\nISL28177 (Single): SOT23-5, SOIC-8\n\nApplications:\nPrecision Active Filters\nMedical and A

In [4]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# Load the BAAI small model
model = SentenceTransformer('BAAI/bge-small-en')
# Generate embeddings with progress bar
document_embeddings = []
for doc in tqdm(documents, desc="Generating embeddings"):
    embedding = model.encode(doc["content"], normalize_embeddings=True)
    document_embeddings.append(embedding)

# Add embeddings to the documents dictionary
for i, doc in enumerate(documents):
    doc["embedding"] = document_embeddings[i]

print("Embeddings generated successfully.")

/Users/jayeshbhadane/Desktop/Vettura AI/GenAI/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-02-11 19:16:53.623403: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Generating embeddings: 100%|██████████| 5/5 [00:05<00:00,  1.18s/it]

Embeddings generated successfully.


In [5]:
import numpy as np

# Convert document_embeddings to a numpy array and get its shape
embeddings_array = np.array(document_embeddings)
print(embeddings_array.shape)

(5, 384)


In [6]:
import faiss
import numpy as np

# Convert embeddings to a numpy array
embeddings_array = np.array([doc["embedding"] for doc in documents]).astype("float32")

# Create a FAISS index
dimension = embeddings_array.shape[1]  # Dimension of the embeddings
index = faiss.IndexFlatL2(dimension)  # Using L2 distance for similarity search

# Add embeddings to the index
index.add(embeddings_array)

print("Embeddings stored in FAISS index.")

Embeddings stored in FAISS index.


In [7]:
import pickle

# Save the FAISS index
faiss.write_index(index, "faiss_index.index")

# Save the documents with metadata
with open("documents.pkl", "wb") as f:
    pickle.dump(documents, f)

print("FAISS index and documents saved to disk.")

FAISS index and documents saved to disk.


In [8]:
#  Load the documents with metadata
with open("documents.pkl", "rb") as f:
    documents = pickle.load(f)

print("FAISS index and documents loaded successfully.")

FAISS index and documents loaded successfully.


In [9]:
def hybrid_search(query, part_name=None):
    if part_name:
        # If part name is provided, find the corresponding document
        for doc in documents:
            if part_name.lower() in doc["filename"].lower():
                return doc
        return None  # Return None if no matching document is found
    else:
        # If no part name is provided, perform semantic search
        query_embedding = model.encode([query], normalize_embeddings=True).astype("float32")
        distances, indices = index.search(query_embedding, k=1)  # Retrieve the top 1 document
        return documents[indices[0][0]]  # Return the most relevant document

In [11]:
# Example 1: Query with part name
part_name = "EL4543"  # Replace with an actual part name from your documents
query_with_part = "What are features of " + part_name + "?"
result_with_part = hybrid_search(query_with_part, part_name)
print("Result with part name:", result_with_part["filename"] if result_with_part else "No matching document found.")

# Example 2: Query without part name
query_without_part = "Which part is used in the applications of Differential line driver"
result_without_part = hybrid_search(query_without_part)
print("Result without part name:", result_without_part["filename"])

Result with part name: EL4543.txt
Result without part name: EL5378.txt


In [18]:
from groq import Groq
import dotenv

dotenv.load_dotenv()

# Initialize Groq API client
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [49]:
def generate_answer_with_groq(query, document):
    # Prepare the messages for Groq's model
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer the user's question based on the provided document."},
        {"role": "user", "content": f"Document: {document['content']}\n\nQuestion: {query}\nAnswer:"}
    ]

    # Send the messages to Groq's model
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # Use the appropriate Groq model
        messages=messages,
        max_tokens=150,  # Adjust based on your needs
        temperature=1.2,  # Adjust for creativity vs. accuracy
    )

    # Extract and return the answer
    return response.choices[0].message.content.strip()

In [ ]:
query = "List all the applications of CA3130?"
document = hybrid_search(query, part_name="CA3130")  # Add in your part name
if document:
    answer = generate_answer_with_groq(query, document)
    print("Groq's response:", answer)
else:
    print("No relevant document found.")

Answer: The applications of CA3130 include:

1. Ground-Referenced Single Supply Amplifiers
2. Fast Sample-Hold Amplifiers
3. Long-Duration Timers/Monostables
4. High-Input-Impedance Comparators (Ideal Interface with Digital CMOS)
5. High-Input-Impedance Wideband Amplifiers
6. Voltage Followers (e.g. Follower for Single-Supply D/A Converter)
7. Voltage Regulators (Permits Control of Output Voltage Down to 0V)
8. Peak Detectors
9. Single-Supply Full-Wave Precision Rectifiers
10. Photo-Diode Sensor Amplifiers


In [ ]:
query_without_part = "Can I use ISL28177 for Ground-Referenced Single Supply Amplifiers?"
document_without_part = hybrid_search(query_without_part)
if document_without_part:
    answer_without_part = generate_answer_with_groq(query_without_part, document_without_part)
    print("Groq's response:", answer_without_part)
else:
    print("No relevant document found.")

Groq's response: The document does not explicitly mention "Ground-Referenced Single Supply Amplifiers" as one of the applications. However, it does mention that the amplifier operates over a wide supply range of 6V (±3V) to 40V (±20V), which implies it can be used in single-supply configurations.

Although it is not explicitly stated, the fact that it can operate from a single supply voltage (e.g., 6V to 40V) with respect to ground suggests that it could be suitable for ground-referenced single supply amplifier applications.

To confirm, it would be best to consult the datasheet or contact the manufacturer for more detailed information on using the ISL28177 in a ground-referenced single
